# 7.1 RLHF

RLHF（基于人类反馈的强化学习）是当前大模型对齐的主流方法，通过训练奖励模型来近似人类偏好，再用PPO优化策略模型。

**目的**：使模型输出符合人类偏好

**基本原理**：三步流程——
1. **SFT**：在高质量数据上监督微调
2. **RM**：训练奖励模型学习人类偏好排序
3. **PPO**：用强化学习优化策略模型最大化奖励

**核心公式**：
- 奖励模型：L = -E[log σ(r(x,y_w) - r(x,y_l))]
- PPO目标：maximize E[r(x,y)] - β·KL(π_θ || π_ref)
- KL惩罚防止模型偏离参考策略太远

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class RewardModel(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_model, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.encoder(x).squeeze(-1)

reward_model = RewardModel()
d_model = 128
x_prompt = torch.randn(4, d_model)
y_chosen = torch.randn(4, d_model) + 0.5
y_rejected = torch.randn(4, d_model) - 0.5

r_chosen = reward_model(y_chosen)
r_rejected = reward_model(y_rejected)

bradley_terry_loss = -F.logsigmoid(r_chosen - r_rejected).mean()

print('=== RLHF: Reward Model Training ===')
print(f'Chosen rewards: {r_chosen.detach().tolist()}')
print(f'Rejected rewards: {r_rejected.detach().tolist()}')
print(f'Bradley-Terry loss: {bradley_terry_loss.item():.4f}')
print(f'\nLoss = -E[log σ(r_chosen - r_rejected)]')
print(f'When r_chosen > r_rejected, loss is low (correct preference).')
print(f'\nKey: RM learns to assign higher scores to human-preferred outputs.')

## PPO优化阶段

使用PPO算法优化策略模型，最大化奖励同时通过KL散度惩罚防止偏离参考策略太远。

**PPO-Clip核心**：
- ratio = π_θ(a|s) / π_ref(a|s)
- L_clip = min(ratio·A, clip(ratio, 1-ε, 1+ε)·A)
- 总目标 = L_clip - β·KL(π_θ || π_ref)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class PolicyModel(nn.Module):
    def __init__(self, d_model=128, vocab_size=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 256), nn.ReLU(),
            nn.Linear(256, vocab_size)
        )
    def forward(self, x):
        return F.log_softmax(self.net(x), dim=-1)

policy = PolicyModel()
ref_policy = PolicyModel()
ref_policy.load_state_dict(policy.state_dict())
reward_model = RewardModel()

x = torch.randn(4, 128)
old_log_probs = policy(x)
actions = old_log_probs.exp().multinomial(1).squeeze(-1)
old_action_log_probs = old_log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)

with torch.no_grad():
    action_emb = F.one_hot(actions, 100).float()
    rewards = reward_model(action_emb)
    ref_log_probs = ref_policy(x)
    ref_action_log_probs = ref_log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)

new_log_probs = policy(x)
new_action_log_probs = new_log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)

ratio = (new_action_log_probs - old_action_log_probs).exp()
kl_penalty = (new_action_log_probs - ref_action_log_probs).mean()

advantages = rewards - rewards.mean()
epsilon = 0.2
surr1 = ratio * advantages
surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
ppo_loss = -torch.min(surr1, surr2).mean()
beta = 0.1
total_loss = ppo_loss + beta * kl_penalty

print('=== RLHF: PPO Optimization ===')
print(f'Rewards: {rewards.tolist()}')
print(f'Advantages: {advantages.tolist()}')
print(f'PPO ratio: {ratio.detach().tolist()}')
print(f'PPO loss: {ppo_loss.item():.4f}')
print(f'KL penalty: {kl_penalty.item():.4f}')
print(f'Total loss: {total_loss.item():.4f}')
print(f'\nKey: PPO balances reward maximization with staying close to reference policy.')